# ResNet18 last-block Grad-CAM v1

Bu notebook yalnız dondurulmuş 37 test görüntüsünde post-hoc Grad-CAM üretir. Eğitim, model seçimi veya test metriği hesabı yapmaz. Yanlış örneklerde tahmin edilen ve gerçek sınıf hedefleri ayrı ayrı gösterilir. Isı haritası tıbbi lokalizasyon veya nedensellik kanıtı değildir.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

assert 'google.colab' in sys.modules, 'Bu notebook Google Colab için hazırlanmıştır.'
from google.colab import drive
drive.mount('/content/drive')
print('Drive bağlandı.')

REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
ZIP_PATH = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer.zip')
LOCAL_DATA_BASE = Path('/content/NNGA_DATA')
CHECKPOINT = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_lastblock_finetune_v1/best_model.pth')
PREDICTIONS = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_lastblock_finetune_v1/test_predictions.csv')
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_lastblock_gradcam_v1')
REVIEW_ZIP = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_lastblock_gradcam_v1_review_package.zip')

if PROJECT_ROOT.exists():
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'checkout', 'main'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
print('Git commit:', commit)

assert ZIP_PATH.is_file(), f'Dataset ZIP bulunamadı: {ZIP_PATH}'
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert free_gib >= 10, f'Colab yerel diskinde en az 10 GiB boş alan gerekli; mevcut={free_gib:.2f}'
marker = LOCAL_DATA_BASE / '.multi_cancer_extract_complete'
if not marker.is_file():
    if LOCAL_DATA_BASE.exists():
        shutil.rmtree(LOCAL_DATA_BASE)
    LOCAL_DATA_BASE.mkdir(parents=True)
    print(f'ZIP çıkarılıyor: {ZIP_PATH.stat().st_size / (1024 ** 3):.2f} GiB')
    started = time.time()
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(LOCAL_DATA_BASE)
    marker.write_text('complete\n', encoding='utf-8')
    print(f'ZIP çıkarma tamamlandı: {(time.time() - started) / 60:.1f} dakika')

candidates = []
for path in LOCAL_DATA_BASE.rglob('*'):
    if path.is_dir() and all((path / name).is_dir() for name in ('ALL', 'Brain Cancer', 'Breast Cancer')):
        candidates.append(path)
assert len(candidates) == 1, f'Beklenen tek dataset kökü bulunamadı: {candidates}'
DATASET_ROOT = candidates[0]
CONFIG = PROJECT_ROOT / 'config' / 'gradcam_resnet18_lastblock_v1.json'
SAMPLE_MANIFEST = PROJECT_ROOT / 'research' / 'gradcam' / 'resnet18_lastblock_gradcam_v1' / 'sample_manifest.csv'
SPLIT_MANIFEST = PROJECT_ROOT / 'research' / 'splits' / 'practical_leakage_reduced_split_v1.csv'
RUNNER = PROJECT_ROOT / 'src' / 'run_gradcam_audit.py'
print('Dataset:', DATASET_ROOT)
print('Checkpoint:', CHECKPOINT)
print('Kalıcı çıktı:', OUTPUT_DIR)

In [ ]:
import torch, torchvision, numpy, PIL, matplotlib

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

EXPECTED = {
    RUNNER: '4a72f5e5dc28b0ef25e4195060be69d0fccdecdd44bf6e8ddf680618de89f538',
    CONFIG: '07f7a88c21a6e65ed6d96084d86f886eb47f2f4bc1be5096c32259cb5ca50df2',
    SAMPLE_MANIFEST: '3f797979b4d75b092dfcb4e2348ea1705ed34d35da884ac999c68abef8c87add',
    SPLIT_MANIFEST: '999bf7a9bd1bba9f2957a9db0e13ae72adb3ecb8fdbc1f33cce40b083c1cf171',
    CHECKPOINT: '538f2891075f0870e450cd6aa7eda0a5e06ecb73eefd14a4f6a0cd869759d54d',
    PREDICTIONS: '237f3283b88beeaf0a93f182adab23ad478810b0944245e3023e6cfe885c57bd',
}
for path, expected_hash in EXPECTED.items():
    assert path.is_file(), f'Gerekli dosya bulunamadı: {path}'
    actual_hash = sha256(path)
    assert actual_hash == expected_hash, f'Hash uyuşmazlığı: {path} expected={expected_hash} actual={actual_hash}'

assert torch.cuda.is_available(), 'T4 GPU çalışma zamanı seçilmemiş.'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('NumPy:', numpy.__version__, '| Pillow:', PIL.__version__, '| matplotlib:', matplotlib.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Tüm dondurulmuş girdiler SHA-256 ile doğrulandı.')
print('Örnek sayısı: 37 = 26 doğru kontrol + 11 hata')

In [ ]:
AUTHORIZE_GRADCAM = False
assert AUTHORIZE_GRADCAM, 'Grad-CAM çalışması kilitli; yalnız onaydan sonra True yap.'
assert not (OUTPUT_DIR / 'summary.json').exists(), 'Bu Grad-CAM audit zaten tamamlanmış; yeniden çalıştırma.'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR.parent / 'resnet18_lastblock_gradcam_v1_console.log'
command = [
    sys.executable, '-u', str(RUNNER),
    '--data-dir', str(DATASET_ROOT),
    '--checkpoint', str(CHECKPOINT),
    '--predictions', str(PREDICTIONS),
    '--split-manifest', str(SPLIT_MANIFEST),
    '--sample-manifest', str(SAMPLE_MANIFEST),
    '--config', str(CONFIG),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'cuda',
]
print('Grad-CAM komutu:', ' '.join(command))
with LOG_PATH.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)
        log_handle.flush()
    return_code = process.wait()
assert return_code == 0, f'Grad-CAM işlemi hata verdi; return code={return_code}. Log: {LOG_PATH}'
print('Grad-CAM tamamlandı.')

In [ ]:
from google.colab import files

SUMMARY_PATH = OUTPUT_DIR / 'summary.json'
HASH_MANIFEST_PATH = OUTPUT_DIR / 'hash_manifest.json'
assert SUMMARY_PATH.is_file(), 'summary.json bulunamadı.'
assert HASH_MANIFEST_PATH.is_file(), 'hash_manifest.json bulunamadı.'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
assert summary['status'] == 'resnet18_lastblock_gradcam_v1_complete'
assert summary['sample_count'] == 37
assert summary['correct_control_count'] == 26
assert summary['error_count'] == 11
assert summary['prediction_reproduction_count'] == 37
assert summary['selection_was_frozen_before_heatmaps'] is True
assert summary['panel_count'] == 37
assert summary['contact_sheet_count'] == 7
if REVIEW_ZIP.exists():
    REVIEW_ZIP.unlink()
shutil.make_archive(str(REVIEW_ZIP.with_suffix('')), 'zip', root_dir=OUTPUT_DIR)
print('Durum:', summary['status'])
print('Örnek:', summary['sample_count'])
print('Doğru kontrol:', summary['correct_control_count'])
print('Hata:', summary['error_count'])
print('Tahmin yeniden üretimi:', summary['prediction_reproduction_count'], '/ 37')
print('Panel:', summary['panel_count'], '| temas sayfası:', summary['contact_sheet_count'])
print('Review package:', REVIEW_ZIP)
print('Review package SHA-256:', sha256(REVIEW_ZIP))
files.download(str(REVIEW_ZIP))